# 03 — How ideal is a real gas?

The ideal gas law says $PV = Nk_BT$, i.e. the *compressibility factor*

$$Z = \frac{P}{\rho\,k_B T} = 1$$

for every density and temperature. A Lennard-Jones gas disagrees: at low
temperature attraction pulls the pressure **down** ($Z < 1$), at high
density the atoms' own volume pushes it **up** ($Z > 1$). We measure
$Z(\rho, T)$ directly, then predict the low-density behavior from the pair
potential alone with the second virial coefficient — computed in numpy, no
simulation required.

In [ ]:
%pip install lammps-js matplotlib

## Measure Z across densities and temperatures

One short NVT run per state point (500 atoms, 3D). Each takes a couple of
seconds — the loop prints as it goes:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lammps import lammps

async def measure_Z(rho, T):
    L = (500 / rho) ** (1 / 3)
    lmp = await lammps(output=None)
    lmp.commands_string(f"""
units         lj
region        box block 0 {L:.4f} 0 {L:.4f} 0 {L:.4f}
create_box    1 box
create_atoms  1 random 500 {1000 + int(rho * 100) + int(T * 7)} box overlap 0.9 maxtry 200
mass          1 1.0
pair_style    lj/cut 2.5
pair_coeff    1 1 1.0 1.0 2.5
minimize      1e-4 1e-6 200 2000
reset_timestep 0
velocity      all create {T} 4928459 dist gaussian
fix           1 all nvt temp {T} {T} 0.5
thermo        1000
run           2500
""")
    # average P over a further run
    samples = []
    for _ in range(6):
        lmp.command("run 250")
        samples.append(lmp.get_thermo("press"))
    Tm = lmp.get_thermo("temp")
    lmp.close()
    return np.mean(samples) / (rho * Tm)

temps = [1.5, 3.0, 5.0]
rhos = [0.05, 0.1, 0.2, 0.3, 0.45, 0.6]
Z = {}
for T in temps:
    Z[T] = [await measure_Z(rho, T) for rho in rhos]
    print(f"T = {T}:", " ".join(f"{z:.3f}" for z in Z[T]))

## The second virial coefficient, from numpy

To lowest order in density, $Z = 1 + B_2(T)\,\rho$ with

$$B_2(T) = -2\pi \int_0^\infty \left(e^{-u(r)/k_BT} - 1\right) r^2\,dr,$$

a one-line numerical integral over the pair potential. $B_2 < 0$ means
attraction wins (pressure deficit); it crosses zero at the **Boyle
temperature** $T_B \approx 3.4$, where the gas *looks* ideal far beyond
its right to:

In [ ]:
r = np.linspace(0.01, 8, 4000)
u = 4 * (r**-12 - r**-6)

def B2(T):
    return -2 * np.pi * np.trapezoid((np.exp(-u / T) - 1) * r**2, r)

print("Boyle temperature check: B2(3.0) =", round(B2(3.0), 3),
      " B2(3.42) =", round(B2(3.42), 3), " B2(4.0) =", round(B2(4.0), 3))

rho_line = np.linspace(0, 0.65, 50)
plt.figure(figsize=(6.5, 4))
for T, color in zip(temps, ["tab:blue", "tab:orange", "tab:green"]):
    plt.plot(rhos, Z[T], "o", color=color, label=f"measured, T = {T}")
    plt.plot(rho_line, 1 + B2(T) * rho_line, "--", color=color, lw=1,
             label=f"virial: 1 + B₂ρ  (B₂ = {B2(T):.2f})")
plt.axhline(1, color="k", lw=0.8)
plt.xlabel("density ρ"); plt.ylabel("Z = P / ρT")
plt.legend(fontsize=8); plt.tight_layout(); plt.show()

The virial line nails the low-density limit at every temperature — that's
the whole content of $B_2$ — and the measured points peel away from it as
higher-order terms kick in. At $T = 1.5$ attraction dominates ($Z < 1$
everywhere shown); by $T = 5$ the atoms act like hard spheres and $Z$
climbs above 1 at all densities.

Next: [04 — Reversibility and chaos](04-reversibility-and-chaos.ipynb) —
running Newton's laws backwards.